# 02 - Baseline OpenAI dengan BM25 + Chroma Expanded

**Tujuan**: re-run baseline OpenAI 500 sampel dengan retriever yang menggunakan
BM25 expanded + Chroma expanded. Bandingkan dengan baseline original.

**Setup identik dengan main notebook 02.1 Baseline - OpenAI.ipynb** kecuali:
- BM25 index: `pubmedqa_bm25_expanded.pkl` (acronym expanded)
- Chroma DB: `pubmedqa_chroma_expanded` (re-embedded dengan expanded text)
- Output: `results/bm25expanded_baseline_openai_phase{1,2}.json`

**Estimasi**:
- Phase 1 (500 calls): ~$0.50, ~15 menit
- Phase 2 (500 sampel x ~14 evaluator calls): ~$10, ~25 menit
- Total: ~$11, ~40 menit


In [1]:
import os

# Set env var
# os.environ["OPENAI_API_KEY"] = "<REDACTED — set via shell env or .env file>"

In [2]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime

from openai import OpenAI
from rank_bm25 import BM25Okapi
from datasets import load_dataset
import chromadb

warnings.filterwarnings("ignore")

# Document class for pickle compat
@dataclass
class Document:
    text: str; pubid: str; question: str
    section_label: str; answer: str; decision: str

@dataclass
class RetrievalResult:
    document: Document; score: float; doc_id: int = -1
    bm25_score: float = 0.0; dense_score: float = 0.0
    rrf_score: float = 0.0; reranker_score: float = 0.0

import __main__
__main__.Document = Document

# Set OpenAI API key
# # os.environ["OPENAI_API_KEY"] = "<REDACTED — set via shell env or .env file>"
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY belum di-set.")

print(f"Python: {sys.version.split()[0]} | NumPy: {np.__version__} | chromadb: {chromadb.__version__}")


C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.9 | NumPy: 2.3.5 | chromadb: 1.5.8


In [3]:
# ============================================================
# Konfigurasi
# ============================================================
LLM_MODEL   = "gpt-4.1-mini"
EMBED_MODEL = "text-embedding-3-small"

TOP_K_BM25 = 50; TOP_K_DENSE = 50; TOP_K_RETRIEVAL = 5

DATASET_NAME = "qiaojin/PubMedQA"; DATASET_SUBSET = "pqa_labeled"
MAX_SAMPLES = 500
TEMPERATURE = 0.0; SEED = 42

# Paths - point to EXPANDED indexes
HERE = Path(".").resolve()
NOTEBOOKS_DIR = HERE.parent if HERE.name == "BM25 Expansion" else HERE
PROJECT_ROOT = NOTEBOOKS_DIR.parent
BM25_INDEX_PATH = NOTEBOOKS_DIR / "pubmedqa_bm25_expanded.pkl"
CHROMA_DB_PATH = NOTEBOOKS_DIR / "pubmedqa_chroma_expanded"
RESULTS_DIR = PROJECT_ROOT / "results" / "BM25_Expansion"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_NAME = "bm25expanded_baseline_openai"
PHASE1_PATH = RESULTS_DIR / f"{CONFIG_NAME}_phase1_answers.json"
PHASE2_PATH = RESULTS_DIR / f"{CONFIG_NAME}_phase2_custom.json"

print("Konfigurasi:")
print(f"  LLM            : {LLM_MODEL}")
print(f"  Embedder       : {EMBED_MODEL}")
print(f"  BM25 index     : {BM25_INDEX_PATH.name} (EXPANDED)")
print(f"  Chroma path    : {CHROMA_DB_PATH.name} (EXPANDED)")
print(f"  Sampel         : {MAX_SAMPLES}")
print(f"  Output         : {PHASE1_PATH}")
print(f"  Resume support : YES (jika file output sudah ada)")

assert BM25_INDEX_PATH.exists(), f"BM25 index tidak ditemukan. Run build_expanded_index.py dulu."
assert CHROMA_DB_PATH.exists(), f"Chroma expanded tidak ditemukan. Run notebook 01 dulu."


Konfigurasi:
  LLM            : gpt-4.1-mini
  Embedder       : text-embedding-3-small
  BM25 index     : pubmedqa_bm25_expanded.pkl (EXPANDED)
  Chroma path    : pubmedqa_chroma_expanded (EXPANDED)
  Sampel         : 500
  Output         : C:\Users\Ricky Wijaya\Documents\STI\Semester 8\TA\Code TA\results\BM25_Expansion\bm25expanded_baseline_openai_phase1_answers.json
  Resume support : YES (jika file output sudah ada)


In [4]:
# ============================================================
# Load BM25 expanded + Chroma expanded
# ============================================================
def tokenize_bm25(text):
    return re.sub(r"[^a-zA-Z0-9\s]", " ", text.lower()).split()

with open(BM25_INDEX_PATH, "rb") as f:
    saved = pickle.load(f)
bm25_index = saved["bm25"]
documents = saved["documents"]
print(f"Loaded BM25 expanded: {len(documents)} chunks")

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))
chroma_collection = chroma_client.get_collection(name="pubmedqa_docs_expanded")
print(f"Loaded Chroma expanded: {chroma_collection.count()} vectors")

# Load PubMedQA samples
ds = load_dataset(DATASET_NAME, DATASET_SUBSET)["train"]
pubmedqa_data = ds.select(range(MAX_SAMPLES))
print(f"Loaded {len(pubmedqa_data)} samples (first {MAX_SAMPLES})")


Loaded BM25 expanded: 1706 chunks
Loaded Chroma expanded: 1706 vectors


Loaded 500 samples (first 500)


In [5]:
# ============================================================
# OpenAI client + helper functions
# ============================================================
client = OpenAI(api_key=api_key)

def openai_generate(prompt, max_tokens=300, temperature=TEMPERATURE):
    for attempt in range(5):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature, max_tokens=max_tokens, seed=SEED,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if "429" in err or "rate" in err.lower():
                time.sleep((attempt + 1) * 10)
            elif "500" in err or "502" in err or "503" in err:
                time.sleep((attempt + 1) * 5)
            else:
                raise
    raise RuntimeError("OpenAI gagal setelah 5 retry")


def openai_embed(texts):
    for attempt in range(5):
        try:
            resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
            return [d.embedding for d in resp.data]
        except Exception as e:
            err = str(e)
            if "429" in err or "rate" in err.lower():
                time.sleep((attempt + 1) * 10)
            elif "500" in err or "502" in err or "503" in err:
                time.sleep((attempt + 1) * 5)
            else:
                raise
    raise RuntimeError("OpenAI embed gagal")


# Smoke test
_test = openai_generate("Reply with exactly: OK", max_tokens=5)
print(f"OpenAI test: {_test!r}")


OpenAI test: 'OK'


In [6]:
# ============================================================
# Hybrid retrieval (BM25 expanded + Dense expanded + RRF)
# ============================================================
def retrieve_dense(query, k=TOP_K_DENSE):
    qvec = openai_embed([query])[0]
    res = chroma_collection.query(query_embeddings=[qvec], n_results=k, include=["distances"])
    doc_ids = [int(i) for i in res["ids"][0]]
    distances = res["distances"][0]
    scores = [1.0 - d for d in distances]
    return list(zip(doc_ids, scores))


def retrieve_bm25_raw(query, k=TOP_K_BM25):
    tokens = tokenize_bm25(query)
    scores = bm25_index.get_scores(tokens)
    top_k = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_k]


def reciprocal_rank_fusion(rank_lists, k=60):
    scores = {}
    for rl in rank_lists:
        for rank, doc_id in enumerate(rl):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])


def retrieve_hybrid(query, k_final=TOP_K_RETRIEVAL):
    bm25_results = retrieve_bm25_raw(query, k=TOP_K_BM25)
    dense_results = retrieve_dense(query, k=TOP_K_DENSE)
    bm25_scores = dict(bm25_results); dense_scores = dict(dense_results)
    fused = reciprocal_rank_fusion([
        [d for d, _ in bm25_results], [d for d, _ in dense_results]
    ])
    out = []
    for doc_id, rrf in fused[:k_final]:
        out.append(RetrievalResult(
            document=documents[doc_id], score=rrf, doc_id=doc_id,
            bm25_score=bm25_scores.get(doc_id, 0.0),
            dense_score=dense_scores.get(doc_id, 0.0), rrf_score=rrf,
        ))
    return out


# Smoke test on idx 16
test_q = pubmedqa_data[16]["question"]
test_r = retrieve_hybrid(test_q)
print(f"Query: {test_q}")
print(f"\nTop-{TOP_K_RETRIEVAL} chunks (BM25 expanded + Chroma expanded):")
for i, r in enumerate(test_r, 1):
    src = " *SOURCE*" if r.document.pubid == "11729377" else ""
    print(f"  [{i}] RRF={r.rrf_score:.4f} | BM25={r.bm25_score:6.2f} | Dense={r.dense_score:.3f} | "
          f"({r.document.section_label[:25]}) pubid={r.document.pubid}{src}")


Query: Is there still a need for living-related liver transplantation in children?

Top-5 chunks (BM25 expanded + Chroma expanded):
  [1] RRF=0.0325 | BM25= 38.61 | Dense=0.583 | (SUMMARY BACKGROUND DATA) pubid=11729377 *SOURCE*
  [2] RRF=0.0320 | BM25= 24.62 | Dense=0.589 | (METHODS) pubid=11729377 *SOURCE*
  [3] RRF=0.0320 | BM25= 33.30 | Dense=0.581 | (RESULTS) pubid=11729377 *SOURCE*
  [4] RRF=0.0315 | BM25= 32.15 | Dense=0.575 | (OBJECTIVE) pubid=11729377 *SOURCE*
  [5] RRF=0.0285 | BM25= 16.11 | Dense=0.430 | (BACKGROUND) pubid=21850494


In [7]:
# ============================================================
# Generation prompt + answer (SAMA persis dengan baseline original)
# ============================================================
GENERATION_PROMPT = (
    "You are a medical research assistant. "
    "Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n"
    "Context from medical literature:\n{context}\n\n"
    "Question: {question}\n\n"
    "Instructions:\n"
    "- Carefully read the context and assess whether it supports or refutes the question.\n"
    "- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n"
    "- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n"
    "  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n"
    "  - no    : the evidence refutes or does not support the hypothesis\n"
    "  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n"
    "            others say no), or if the context contains no relevant information at all\n"
    "- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n"
    "  Do NOT use maybe simply because the evidence is limited or not 100% certain.\n\n"
    "Answer:"
)


def generate_answer(query, retrieved):
    context = "\n\n".join(
        f"[{i}] ({r.document.section_label}): {r.document.text}"
        for i, r in enumerate(retrieved, 1)
    )
    return openai_generate(
        GENERATION_PROMPT.format(context=context, question=query),
        max_tokens=300, temperature=TEMPERATURE
    )


def extract_label(answer):
    lines = [l.strip().lower() for l in answer.split("\n") if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r"[^a-z]", "", line)
        if word in ("yes", "no", "maybe"):
            return word
    for label in ("yes", "no", "maybe"):
        if re.search(r"\b" + label + r"\b", answer.lower()):
            return label
    return "maybe"


# Test on idx 16 (case study)
test_ans = generate_answer(test_q, test_r)
print("Generated answer for idx 16 (LRT/SLT case):")
print(test_ans)
print(f"\nPredicted: {extract_label(test_ans)} (GT: {pubmedqa_data[16]['final_decision']})")


Generated answer for idx 16 (LRT/SLT case):
The provided abstracts indicate that the combination of split-liver transplantation (SLT) and living-related liver transplantation (LRT) has abolished deaths on the waiting list, raising the question of whether LRT is still necessary. However, the survival rates and graft function between SLT and LRT are comparable but not identical, with some differences in complications and graft characteristics. This suggests that while SLT is a significant advancement, LRT may still have a role, implying that living-related liver transplantation is still needed in children.

yes

Predicted: yes (GT: yes)


## Phase 1 — Generate jawaban 500 sampel (resumable)

Estimasi 15 menit.


In [8]:
# Resume support
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, "r", encoding="utf-8") as f:
        results_p1 = json.load(f)["results"]
    start_from = len(results_p1)
    print(f"Resume Fase 1: {start_from}/{MAX_SAMPLES} sudah selesai.")
else:
    results_p1, start_from = [], 0
    print(f"Memulai Fase 1: {MAX_SAMPLES} sampel.")

if start_from < MAX_SAMPLES:
    t0 = time.time()
    for i in range(start_from, MAX_SAMPLES):
        s = pubmedqa_data[i]
        q, gt, ref = s["question"], s["final_decision"], s["long_answer"]

        retrieved = retrieve_hybrid(q)
        answer = generate_answer(q, retrieved)
        predicted = extract_label(answer)

        results_p1.append({
            "idx": i, "pubid": str(s["pubid"]), "question": q,
            "ground_truth": gt, "predicted_label": predicted,
            "is_correct": predicted == gt, "answer": answer,
            "contexts": [r.document.text for r in retrieved],
            "context_pubids": [r.document.pubid for r in retrieved],
            "context_sections": [r.document.section_label for r in retrieved],
            "reference": ref,
            "retrieval_scores": [r.bm25_score for r in retrieved],
            "dense_scores": [r.dense_score for r in retrieved],
            "rrf_scores": [r.rrf_score for r in retrieved],
        })

        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, "w", encoding="utf-8") as f:
                json.dump({"config": CONFIG_NAME, "llm_model": LLM_MODEL,
                           "embed_model": EMBED_MODEL,
                           "timestamp": datetime.now().isoformat(),
                           "max_samples": MAX_SAMPLES, "completed": i+1,
                           "results": results_p1}, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc = sum(r["is_correct"] for r in results_p1) / done
            eta = (time.time() - t0) / (i + 1 - start_from) * (MAX_SAMPLES - i - 1) / 60
            print(f"  [{done:3d}/{MAX_SAMPLES}] acc={acc:.1%} | ETA {eta:.1f} mnt")

print(f"\nFase 1 selesai -> {PHASE1_PATH}")


Memulai Fase 1: 500 sampel.
  [ 10/500] acc=50.0% | ETA 23.3 mnt
  [ 20/500] acc=70.0% | ETA 18.3 mnt
  [ 30/500] acc=73.3% | ETA 18.3 mnt
  [ 40/500] acc=72.5% | ETA 18.1 mnt
  [ 50/500] acc=74.0% | ETA 17.4 mnt
  [ 60/500] acc=70.0% | ETA 17.0 mnt
  [ 70/500] acc=72.9% | ETA 16.3 mnt
  [ 80/500] acc=70.0% | ETA 16.1 mnt
  [ 90/500] acc=70.0% | ETA 15.7 mnt
  [100/500] acc=70.0% | ETA 15.2 mnt
  [110/500] acc=70.9% | ETA 14.9 mnt
  [120/500] acc=70.0% | ETA 14.5 mnt
  [130/500] acc=68.5% | ETA 14.1 mnt
  [140/500] acc=68.6% | ETA 13.6 mnt
  [150/500] acc=68.0% | ETA 13.1 mnt
  [160/500] acc=68.8% | ETA 12.7 mnt
  [170/500] acc=69.4% | ETA 12.3 mnt
  [180/500] acc=70.6% | ETA 11.9 mnt
  [190/500] acc=70.5% | ETA 11.5 mnt
  [200/500] acc=70.5% | ETA 11.0 mnt
  [210/500] acc=71.0% | ETA 10.6 mnt
  [220/500] acc=70.5% | ETA 10.3 mnt
  [230/500] acc=70.4% | ETA 10.1 mnt
  [240/500] acc=69.6% | ETA 9.8 mnt
  [250/500] acc=69.6% | ETA 9.5 mnt
  [260/500] acc=69.6% | ETA 9.1 mnt
  [270/500] a

## Phase 1 — Analisis cepat


In [9]:
with open(PHASE1_PATH, "r", encoding="utf-8") as f:
    results_p1 = json.load(f)["results"]

n = len(results_p1)
n_correct = sum(r["is_correct"] for r in results_p1)
print(f"BM25 Expanded — Baseline OpenAI ({n} sampel)")
print("=" * 60)
print(f"Label Accuracy : {n_correct}/{n} = {n_correct/n:.1%}")
print()
print("Per-label accuracy:")
for lbl in ["yes", "no", "maybe"]:
    sub = [r for r in results_p1 if r["ground_truth"] == lbl]
    if sub:
        c = sum(r["is_correct"] for r in sub)
        print(f"  {lbl:>5}: {c}/{len(sub)} = {c/len(sub):.1%}")

# Quick comparison with original baseline
orig_path = PROJECT_ROOT / "results" / "baseline_openai_phase1_answers.json"
if orig_path.exists():
    with open(orig_path, "r", encoding="utf-8") as f:
        orig = json.load(f)["results"]
    orig_acc = sum(r["is_correct"] for r in orig) / len(orig)
    delta = (n_correct/n - orig_acc) * 100
    print(f"\n--- COMPARISON vs baseline original ---")
    print(f"  Original BM25 baseline : {orig_acc:.1%}")
    print(f"  Expanded BM25 baseline : {n_correct/n:.1%}")
    print(f"  Δ (delta)              : {delta:+.1f} pp")

# Check idx 16 specifically
idx16 = next((r for r in results_p1 if r["idx"] == 16), None)
if idx16:
    print(f"\n--- IDX 16 (LRT/SLT case study) ---")
    print(f"  GT: {idx16['ground_truth']} | Pred: {idx16['predicted_label']}")
    print(f"  Source paper sections retrieved:")
    for i, (pubid, section) in enumerate(zip(idx16["context_pubids"], idx16["context_sections"]), 1):
        marker = " <-- SOURCE" if pubid == "11729377" else ""
        print(f"    [{i}] pubid={pubid} ({section}){marker}")


BM25 Expanded — Baseline OpenAI (500 sampel)
Label Accuracy : 364/500 = 72.8%

Per-label accuracy:
    yes: 244/275 = 88.7%
     no: 115/159 = 72.3%
  maybe: 5/66 = 7.6%

--- COMPARISON vs baseline original ---
  Original BM25 baseline : 69.2%
  Expanded BM25 baseline : 72.8%
  Δ (delta)              : +3.6 pp

--- IDX 16 (LRT/SLT case study) ---
  GT: yes | Pred: yes
  Source paper sections retrieved:
    [1] pubid=11729377 (SUMMARY BACKGROUND DATA) <-- SOURCE
    [2] pubid=11729377 (METHODS) <-- SOURCE
    [3] pubid=11729377 (RESULTS) <-- SOURCE
    [4] pubid=11729377 (OBJECTIVE) <-- SOURCE
    [5] pubid=21850494 (BACKGROUND)


## Phase 2 — Custom evaluator 4 metrik (resumable)

Estimasi 25 menit.


In [ ]:
# Custom 4-metric evaluator (sama persis dengan main notebook)
def _split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s.strip() for s in parts if len(s.strip()) >= 15]


def _llm_yes_no(prompt):
    try:
        resp = openai_generate(prompt, max_tokens=10, temperature=0.0)
        return "yes" in resp.lower()[:15]
    except Exception:
        return False


def compute_faithfulness(answer, contexts):
    sents = _split_sentences(answer)
    if not sents: return 0.0
    ctx_text = "\n".join(f"[{i+1}] {c[:400]}" for i, c in enumerate(contexts))
    tmpl = ("Context:\n{ctx}\n\nStatement: {sent}\n\n"
            "Is this statement directly supported by the context above? "
            "Answer with only \"yes\" or \"no\".")
    return sum(_llm_yes_no(tmpl.format(ctx=ctx_text, sent=s)) for s in sents) / len(sents)


def compute_context_recall(reference, contexts):
    sents = _split_sentences(reference)
    if not sents: return 0.0
    ctx_text = "\n".join(f"[{i+1}] {c[:400]}" for i, c in enumerate(contexts))
    tmpl = ("Context:\n{ctx}\n\nStatement: {sent}\n\n"
            "Is this statement supported by the context above? "
            "Answer with only \"yes\" or \"no\".")
    return sum(_llm_yes_no(tmpl.format(ctx=ctx_text, sent=s)) for s in sents) / len(sents)


def compute_answer_relevancy(question, answer):
    sents = _split_sentences(answer)
    if not sents: return 0.0
    tmpl = ("Question: {q}\n\nStatement: {sent}\n\n"
            "Is this statement relevant to answering the question above? "
            "Answer with only \"yes\" or \"no\".")
    return sum(_llm_yes_no(tmpl.format(q=question, sent=s)) for s in sents) / len(sents)


def compute_context_precision(question, contexts, reference):
    if not contexts: return 0.0
    tmpl = ("Question: {q}\n\nGround truth answer: {ref}\n\nRetrieved context: {ctx}\n\n"
            "Does this context contain information useful for correctly answering "
            "the question based on the ground truth? Answer with only \"yes\" or \"no\".")
    relevance = [1 if _llm_yes_no(tmpl.format(q=question, ref=reference[:300], ctx=c[:400])) else 0
                 for c in contexts]
    total_rel = sum(relevance)
    if total_rel == 0: return 0.0
    prec_sum = 0.0; rel_count = 0
    for k, rel in enumerate(relevance):
        if rel:
            rel_count += 1; prec_sum += rel_count / (k + 1)
    return prec_sum / total_rel


def evaluate_custom(q, ans, ctx, ref):
    return {
        "faithfulness": compute_faithfulness(ans, ctx),
        "context_recall": compute_context_recall(ref, ctx),
        "answer_relevancy": compute_answer_relevancy(q, ans),
        "context_precision": compute_context_precision(q, ctx, ref),
    }


# Smoke test
_r = evaluate_custom(
    "Does aspirin prevent heart attacks?",
    "Aspirin helps prevent heart attacks. It reduces clotting.",
    ["Aspirin reduces blood clotting and is used for heart attack prevention."],
    "Aspirin is used for heart attack prevention by reducing blood clotting.",
)
print("Smoke test 4 metrics:", {k: f"{v:.3f}" for k, v in _r.items()})


In [ ]:
# ============================================================
# Phase 2 evaluator loop (resumable)
# ============================================================
REQ = ["faithfulness", "context_recall", "answer_relevancy", "context_precision"]

with open(PHASE1_PATH, "r", encoding="utf-8") as f:
    p1_results = json.load(f)["results"][:MAX_SAMPLES]

if PHASE2_PATH.exists():
    with open(PHASE2_PATH, "r", encoding="utf-8") as f:
        p2_results = json.load(f)["results"]
    done = {r["idx"] for r in p2_results if all(m in r for m in REQ)}
else:
    p2_results, done = [], set()

remaining = [r for r in p1_results if r["idx"] not in done]
print(f"Phase 2: {len(done)}/{MAX_SAMPLES} done, {len(remaining)} remaining.")

if remaining:
    t0 = time.time()
    for i, r in enumerate(remaining):
        scores = evaluate_custom(r["question"], r["answer"], r["contexts"], r["reference"])
        p2_results.append({
            "idx": r["idx"], "ground_truth": r["ground_truth"],
            "predicted_label": r["predicted_label"], "is_correct": r["is_correct"],
            **scores,
        })
        if (i + 1) % 10 == 0 or i == len(remaining) - 1:
            with open(PHASE2_PATH, "w", encoding="utf-8") as f:
                json.dump({"config": CONFIG_NAME,
                           "timestamp": datetime.now().isoformat(),
                           "max_samples": MAX_SAMPLES,
                           "metrics": REQ, "evaluator": "custom_zero_nan_4metrics",
                           "results": p2_results}, f, indent=2, ensure_ascii=False)
            done_n = i + 1
            avg = {m: sum(x[m] for x in p2_results) / len(p2_results) for m in REQ}
            eta = (time.time() - t0) / (i + 1) * (len(remaining) - i - 1) / 60
            print(f"  [{done_n:3d}/{len(remaining)}] "
                  f"f={avg['faithfulness']:.3f} cr={avg['context_recall']:.3f} "
                  f"ar={avg['answer_relevancy']:.3f} cp={avg['context_precision']:.3f} | ETA {eta:.1f}m")

print(f"\nFase 2 selesai -> {PHASE2_PATH}")


## Comparison Final: BM25 Expanded vs Original


In [10]:
# ============================================================
# Comparison final
# ============================================================
def summarize(p1_path, p2_path, label):
    with open(p1_path, "r", encoding="utf-8") as f:
        p1 = json.load(f)["results"]
    with open(p2_path, "r", encoding="utf-8") as f:
        p2 = json.load(f)["results"]
    n = len(p1)
    acc = sum(r["is_correct"] for r in p1) / n
    accs_per_label = {}
    for lbl in ["yes", "no", "maybe"]:
        sub = [r for r in p1 if r["ground_truth"] == lbl]
        accs_per_label[lbl] = sum(r["is_correct"] for r in sub) / len(sub) if sub else 0
    bal_acc = sum(accs_per_label.values()) / 3
    metrics = {m: sum(r[m] for r in p2) / len(p2) for m in
               ["faithfulness", "context_recall", "answer_relevancy", "context_precision"]}
    return {"label": label, "n": n, "acc": acc, "bal_acc": bal_acc,
            "yes_acc": accs_per_label["yes"], "no_acc": accs_per_label["no"],
            "maybe_acc": accs_per_label["maybe"], **metrics}


orig = summarize(
    PROJECT_ROOT / "results" / "baseline_openai_phase1_answers.json",
    PROJECT_ROOT / "results" / "baseline_openai_phase2_custom.json",
    "Original BM25"
)
expanded = summarize(PHASE1_PATH, PHASE2_PATH, "Expanded BM25 + Chroma")

print("=" * 70)
print("COMPARISON: BM25 Original vs BM25+Chroma Expanded")
print("=" * 70)
print(f"{'Metric':<22} {'Original':>12} {'Expanded':>12} {'Δ (pp)':>10}")
print("-" * 70)

for key, name in [
    ("acc", "Overall accuracy"),
    ("bal_acc", "Balanced accuracy"),
    ("yes_acc", "Acc yes class"),
    ("no_acc", "Acc no class"),
    ("maybe_acc", "Acc maybe class"),
    ("faithfulness", "Faithfulness"),
    ("context_recall", "Context recall"),
    ("answer_relevancy", "Answer relevancy"),
    ("context_precision", "Context precision"),
]:
    delta = (expanded[key] - orig[key]) * 100
    arrow = " ↑" if delta > 0.5 else (" ↓" if delta < -0.5 else " ·")
    print(f"  {name:<20} {orig[key]:>11.3f}  {expanded[key]:>11.3f}  {delta:+8.2f}{arrow}")

# Check idx 16 specifically
idx16_orig = next((r for r in json.load(open(PROJECT_ROOT/"results"/"baseline_openai_phase1_answers.json"))["results"] if r["idx"] == 16), None)
idx16_exp = next((r for r in json.load(open(PHASE1_PATH))["results"] if r["idx"] == 16), None)
if idx16_orig and idx16_exp:
    print()
    print("=" * 70)
    print("IDX 16 (LRT/SLT case study) BEFORE vs AFTER expansion")
    print("=" * 70)
    print(f"  GT: {idx16_orig['ground_truth']}")
    print(f"  Original   pred: {idx16_orig['predicted_label']} (correct: {idx16_orig['is_correct']})")
    print(f"  Expanded   pred: {idx16_exp['predicted_label']} (correct: {idx16_exp['is_correct']})")
    print(f"\n  Source paper sections retrieved (Original):")
    if "context_pubids" in idx16_orig:
        for i, (pid, sec) in enumerate(zip(idx16_orig["context_pubids"], idx16_orig.get("context_sections", [])), 1):
            mark = " <-- SOURCE" if pid == "11729377" else ""
            print(f"    [{i}] pubid={pid}{mark}")
    print(f"  Source paper sections retrieved (Expanded):")
    for i, (pid, sec) in enumerate(zip(idx16_exp["context_pubids"], idx16_exp["context_sections"]), 1):
        mark = " <-- SOURCE" if pid == "11729377" else ""
        print(f"    [{i}] pubid={pid} ({sec}){mark}")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Ricky Wijaya\\Documents\\STI\\Semester 8\\TA\\Code TA\\results\\BM25_Expansion\\bm25expanded_baseline_openai_phase2_custom.json'

## Hasil interpretasi

Lihat tabel comparison di atas. Patokan untuk klaim laporan TA:

- **Δ overall accuracy** > +2pp = signifikan (worth claiming)
- **Δ overall accuracy** -1 sampai +1 pp = neutral (acronym tidak jadi bottleneck utama)
- **Δ overall accuracy** < -1 pp = noise penalty melebihi gain (perlu investigasi false positive)

**Per-class analysis penting**: kalau gain overall kecil tapi maybe accuracy naik banyak,
itu juga klaim valid karena maybe paling underrepresented di prediksi.

**Sample idx 16**: kalau pred berubah dari `maybe` ke `yes` (correct), itu validasi
bahwa acronym expansion betul-betul mempengaruhi inference LLM untuk kasus konkrit
yang kita drill-down.
